# Booster K1: train a walking policy and put it on the robot (v2, robot version)

This notebook trains a walking policy for the Booster K1 in **mjlab** (MuJoCo Warp on the GPU), then exports it as a file that **Booster's own deploy tool (`booster_deploy`)** can run, first in simulation on your Mac, then on the real K1.

The policy uses exactly the same inputs, outputs, joint order, default pose and motor gains as Booster's built-in `k1_walk` policy, so no conversion is needed on the robot side. The export step checks this automatically.

**v2 changes (after v1 shook on the real robot):** random motor command delay (0-20 ms), motor stiffness/damping, body mass, joint friction and armature randomized per robot, random start poses, more IMU noise, stronger penalties on jittery actions, and slightly higher steps. Runs are saved as `k1_velocity_v2`, separate from v1.

**Before you start:** *Runtime → Change runtime type → GPU* (A100 is fastest; L4 also works).

| Step | Time |
|---|---|
| 1. Setup | ~5 min |
| 2. Smoke test | ~2 min |
| 3. Training (4000 iterations, ~390M steps) | about 70 min on an A100 |
| 4. Export + video | ~3 min |

Checkpoints are saved to your Google Drive, so a disconnect doesn't lose progress (use the *resume* cell).

In [1]:
!nvidia-smi

Wed Nov 11 09:15:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   33C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Setup

In [2]:
%cd /content
!git clone -q --depth 1 https://github.com/BoosterRobotics/booster_assets.git || echo "already cloned"
!pip -q install -e booster_assets
!pip -q install mjlab "imageio[ffmpeg]" tensorboard
!mkdir -p /content/k1walk

/content
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for booster_assets (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [3]:
# Save training runs on Google Drive so they survive disconnects.
from google.colab import drive
drive.mount('/content/drive')
import os
LOG_ROOT = '/content/drive/MyDrive/k1_walk/logs'
os.makedirs(LOG_ROOT, exist_ok=True)
print('Runs will be saved in', LOG_ROOT)

Mounted at /content/drive
Runs will be saved in /content/drive/MyDrive/k1_walk/logs


### The task, training and export code
`k1_task.py` defines the K1 walking task (robot model, observations, rewards, randomization). Edit the reward weights or command ranges here if you want to experiment.

In [4]:
%%writefile /content/k1walk/k1_task.py
"""Booster K1 velocity-tracking (walking) task for mjlab.

The policy is set up to match Booster's own deploy tool (booster_deploy, task "k1_walk")
exactly, so a trained policy can run on the real K1 without changes:

  * controls the 20 body joints (head stays at its default), 50 Hz (0.005 s x 4)
  * target = default pose + 0.25 * action, PD gains/effort limits from booster_deploy
  * actor observation per frame (69): base angular velocity (3), projected gravity (3),
    velocity command (3), joint pos - default (20), 0.1 * joint vel (20), last action (20)
  * 10 frames of history (690 inputs)

Importing this module registers the task "Mjlab-Velocity-Flat-Booster-K1".

v2 (robot version): adds what the real motors do that the simulation didn't: 0-20 ms command
delay, +-15% stiffness / -40..+60% damping, +-10% body mass, joint friction and armature,
random start poses, more sensor noise, and stronger penalties on jittery actions.
"""

from __future__ import annotations

import math
import os

import mujoco
from booster_assets import BOOSTER_ASSETS_DIR

from mjlab.actuator import BuiltinPositionActuatorCfg
from mjlab.entity import EntityArticulationInfoCfg, EntityCfg
from mjlab.envs import ManagerBasedRlEnvCfg
from mjlab.envs import mdp as envs_mdp
from mjlab.envs.mdp import dr
from mjlab.managers.event_manager import EventTermCfg
from mjlab.managers.reward_manager import RewardTermCfg
from mjlab.envs.mdp.actions import JointPositionActionCfg
from mjlab.managers.curriculum_manager import CurriculumTermCfg
from mjlab.managers.observation_manager import ObservationGroupCfg, ObservationTermCfg
from mjlab.managers.scene_entity_config import SceneEntityCfg
from mjlab.rl import RslRlModelCfg, RslRlOnPolicyRunnerCfg, RslRlPpoAlgorithmCfg
from mjlab.sensor import ContactMatch, ContactSensorCfg, RingPatternCfg, ObjRef, TerrainHeightSensorCfg
from mjlab.tasks.registry import register_mjlab_task
from mjlab.tasks.velocity import mdp
from mjlab.tasks.velocity.mdp import UniformVelocityCommandCfg
from mjlab.tasks.velocity.rl import VelocityOnPolicyRunner
from mjlab.tasks.velocity.velocity_env_cfg import make_velocity_env_cfg
from mjlab.utils.noise import UniformNoiseCfg as Unoise

TASK_ID = "Mjlab-Velocity-Flat-Booster-K1"
K1_XML = os.path.join(BOOSTER_ASSETS_DIR, "robots/K1/K1_22dof.xml")

# --- Values copied from booster_deploy/tasks/locomotion/robots/k1/__init__.py (k1_walk) ----------
# booster_deploy joint order (all 22 joints).
DEPLOY_JOINTS = [
  "aahead_yaw_joint", "aahead_pitch_joint",
  "aaleft_shoulder_pitch_joint", "left_shoulder_roll_joint", "left_elbow_pitch_joint", "left_elbow_yaw_joint",
  "aaright_shoulder_pitch_joint", "right_shoulder_roll_joint", "right_elbow_pitch_joint", "right_elbow_yaw_joint",
  "left_hip_pitch_joint", "left_hip_roll_joint", "left_hip_yaw_joint",
  "left_knee_pitch_joint", "left_ankle_pitch_joint", "left_ankle_roll_joint",
  "right_hip_pitch_joint", "right_hip_roll_joint", "right_hip_yaw_joint",
  "right_knee_pitch_joint", "right_ankle_pitch_joint", "right_ankle_roll_joint",
]
DEFAULT_POS = [0, 0, 0.2, -1.25, 0, -0.5, 0.2, 1.25, 0, 0.5,
               -0.15, 0, 0, 0.3, -0.15, 0, -0.15, 0, 0, 0.3, -0.15, 0]
STIFFNESS = [4.0, 4.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0,
             100.0, 100.0, 100.0, 100.0, 65.0, 65.0, 100.0, 100.0, 100.0, 100.0, 65.0, 65.0]
DAMPING = [1.0, 1.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0,
           2.0, 2.0, 2.0, 2.0, 1.0, 1.0, 2.0, 2.0, 2.0, 2.0, 1.0, 1.0]
EFFORT = [6.0, 6.0, 14.0, 14.0, 14.0, 14.0, 14.0, 14.0, 14.0, 14.0,
          30.0, 20.0, 15.0, 35.0, 24.0, 15.0, 30.0, 20.0, 15.0, 35.0, 24.0, 15.0]
# The 20 joints the policy controls, in booster_deploy's policy order.
POLICY_JOINTS = [
  "aaleft_shoulder_pitch_joint", "aaright_shoulder_pitch_joint",
  "left_hip_pitch_joint", "right_hip_pitch_joint",
  "left_shoulder_roll_joint", "right_shoulder_roll_joint",
  "left_hip_roll_joint", "right_hip_roll_joint",
  "left_elbow_pitch_joint", "right_elbow_pitch_joint",
  "left_hip_yaw_joint", "right_hip_yaw_joint",
  "left_elbow_yaw_joint", "right_elbow_yaw_joint",
  "left_knee_pitch_joint", "right_knee_pitch_joint",
  "left_ankle_pitch_joint", "right_ankle_pitch_joint",
  "left_ankle_roll_joint", "right_ankle_roll_joint",
]
ACTION_SCALE = 0.25
JOINT_VEL_SCALE = 0.1
HISTORY = 10
BASE_HEIGHT = 0.53
# Motor command delay, in physics steps of 0.005 s: 0..4 = 0..20 ms.
DELAY_MAX_STEPS = 4
# ------------------------------------------------------------------------------------------------

POLICY_JOINT_REGEX = tuple(f"^{n}$" for n in POLICY_JOINTS)
# Bodies with real mass/inertia (camera frames and tiny ankle cross links excluded).
MASS_BODIES = (r"^trunk$", r"^aahead_.*_link$", r"^aa(left|right)_shoulder_pitch_link$",
               r"^(left|right)_(shoulder_roll|elbow_pitch|elbow_yaw)_link$",
               r"^(left|right)_(hip_pitch|hip_roll|hip_yaw|knee_pitch|ankle_roll)_link$")
FOOT_GEOMS = ("left_foot_collision", "right_foot_collision")
FOOT_SITES = ("left_foot", "right_foot")


def get_k1_spec() -> mujoco.MjSpec:
  """Booster's K1 model, adapted for mjlab: no ground/actuators/old sensors, named foot
  geoms, foot sites, and the IMU sensors mjlab's velocity task expects."""
  spec = mujoco.MjSpec.from_file(K1_XML)
  for geom in list(spec.geoms):
    if geom.name == "ground":
      spec.delete(geom)
  for light in list(spec.lights):
    spec.delete(light)
  for act in list(spec.actuators):
    spec.delete(act)
  for sensor in list(spec.sensors):
    spec.delete(sensor)
  for side in ("left", "right"):
    foot = spec.body(f"{side}_ankle_roll_link")
    for geom in foot.geoms:
      if geom.type == mujoco.mjtGeom.mjGEOM_BOX:
        geom.name = f"{side}_foot_collision"
        geom.friction[0] = 0.8
    foot.add_site(name=f"{side}_foot", pos=[0.026, 0.0, -0.038], size=[0.01, 0.01, 0.01])
  spec.add_sensor(name="imu_ang_vel", type=mujoco.mjtSensor.mjSENS_GYRO,
                  objtype=mujoco.mjtObj.mjOBJ_SITE, objname="imu")
  spec.add_sensor(name="imu_lin_vel", type=mujoco.mjtSensor.mjSENS_VELOCIMETER,
                  objtype=mujoco.mjtObj.mjOBJ_SITE, objname="imu")
  spec.add_sensor(name="root_angmom", type=mujoco.mjtSensor.mjSENS_SUBTREEANGMOM,
                  objtype=mujoco.mjtObj.mjOBJ_BODY, objname="trunk")
  return spec


def get_k1_robot_cfg() -> EntityCfg:
  actuators = tuple(
    BuiltinPositionActuatorCfg(
      target_names_expr=(f"^{name}$",), stiffness=kp, damping=kd, effort_limit=eff,
      delay_min_lag=0, delay_max_lag=DELAY_MAX_STEPS)
    for name, kp, kd, eff in zip(DEPLOY_JOINTS, STIFFNESS, DAMPING, EFFORT)
  )
  return EntityCfg(
    init_state=EntityCfg.InitialStateCfg(
      pos=(0.0, 0.0, BASE_HEIGHT),
      joint_pos={f"^{n}$": float(p) for n, p in zip(DEPLOY_JOINTS, DEFAULT_POS)},
      joint_vel={".*": 0.0},
    ),
    spec_fn=get_k1_spec,
    articulation=EntityArticulationInfoCfg(actuators=actuators, soft_joint_pos_limit_factor=0.9),
  )


def k1_flat_env_cfg(play: bool = False) -> ManagerBasedRlEnvCfg:
  cfg = make_velocity_env_cfg()
  cfg.scene.entities = {"robot": get_k1_robot_cfg()}

  # Flat ground only.
  cfg.scene.terrain.terrain_type = "plane"
  cfg.scene.terrain.terrain_generator = None
  cfg.scene.sensors = tuple(s for s in (cfg.scene.sensors or ()) if s.name != "terrain_scan")
  cfg.terminations.pop("out_of_terrain_bounds", None)
  cfg.curriculum.pop("terrain_levels", None)
  cfg.sim.njmax = 300
  cfg.sim.nconmax = None
  cfg.sim.mujoco.ccd_iterations = 50
  cfg.sim.contact_sensor_maxmatch = 64

  for sensor in cfg.scene.sensors or ():
    if sensor.name == "foot_height_scan":
      assert isinstance(sensor, TerrainHeightSensorCfg)
      sensor.frame = tuple(ObjRef(type="site", name=s, entity="robot") for s in FOOT_SITES)
      sensor.pattern = RingPatternCfg.single_ring(radius=0.03, num_samples=6)
  cfg.scene.sensors = (cfg.scene.sensors or ()) + (
    ContactSensorCfg(
      name="feet_ground_contact",
      primary=ContactMatch(mode="subtree", pattern=r"^(left_ankle_roll_link|right_ankle_roll_link)$",
                           entity="robot"),
      secondary=ContactMatch(mode="body", pattern="terrain"),
      fields=("found", "force"), reduce="netforce", num_slots=1, track_air_time=True,
    ),
  )

  # Actions: the 20 policy joints, scale 0.25 around the default pose (same as booster_deploy).
  cfg.actions["joint_pos"] = JointPositionActionCfg(
    entity_name="robot", actuator_names=POLICY_JOINT_REGEX, scale=ACTION_SCALE,
    use_default_offset=True,
  )

  # Actor observations: exactly what booster_deploy computes on the robot.
  policy_joints = SceneEntityCfg("robot", joint_names=POLICY_JOINT_REGEX)
  cfg.observations["actor"] = ObservationGroupCfg(
    terms={
      "base_ang_vel": ObservationTermCfg(
        func=mdp.builtin_sensor, params={"sensor_name": "robot/imu_ang_vel"},
        noise=Unoise(n_min=-0.3, n_max=0.3)),
      "projected_gravity": ObservationTermCfg(
        func=mdp.projected_gravity, noise=Unoise(n_min=-0.05, n_max=0.05)),
      "command": ObservationTermCfg(
        func=mdp.generated_commands, params={"command_name": "twist"}),
      "joint_pos": ObservationTermCfg(
        func=mdp.joint_pos_rel, params={"biased": True, "asset_cfg": policy_joints},
        noise=Unoise(n_min=-0.01, n_max=0.01)),
      "joint_vel": ObservationTermCfg(
        func=mdp.joint_vel_rel, params={"asset_cfg": policy_joints},
        noise=Unoise(n_min=-1.5, n_max=1.5), scale=JOINT_VEL_SCALE),
      "actions": ObservationTermCfg(func=mdp.last_action),
    },
    concatenate_terms=True,
    enable_corruption=not play,
    history_length=HISTORY,
    flatten_history_dim=True,
  )
  # Critic: privileged info (true velocity, contacts), no history, no terrain scan.
  critic = cfg.observations["critic"]
  critic.terms.pop("height_scan", None)

  # Commands sized for the K1 (booster_deploy clamps to +1.6/-0.3 m/s, 0.3 m/s side, 1.8 rad/s).
  twist = cfg.commands["twist"]
  assert isinstance(twist, UniformVelocityCommandCfg)
  twist.ranges.lin_vel_x = (-0.3, 0.6)
  twist.ranges.lin_vel_y = (-0.3, 0.3)
  twist.ranges.ang_vel_z = (-0.8, 0.8)
  twist.viz.z_offset = 0.8
  cfg.curriculum["command_vel"] = CurriculumTermCfg(
    func=mdp.commands_vel,
    params={"command_name": "twist", "velocity_stages": [
      {"step": 0, "lin_vel_x": (-0.3, 0.6), "ang_vel_z": (-0.8, 0.8)},
      {"step": 1500 * 24, "lin_vel_x": (-0.3, 1.0), "ang_vel_z": (-1.0, 1.0)},
    ]},
  )

  # Events / randomization (helps transfer to the real robot).
  cfg.events["foot_friction"].params["asset_cfg"].geom_names = FOOT_GEOMS
  cfg.events["base_com"].params["asset_cfg"].body_names = ("trunk",)
  cfg.events["reset_base"].params["pose_range"]["z"] = (0.0, 0.02)
  # Start from slightly different poses/speeds (e.g. when taking over from another controller).
  cfg.events["reset_robot_joints"].params["position_range"] = (-0.1, 0.1)
  cfg.events["reset_robot_joints"].params["velocity_range"] = (-0.5, 0.5)
  # The real motors and body are never exactly the model: randomize them per robot.
  cfg.events["pd_gains"] = EventTermCfg(
    mode="startup", func=dr.pd_gains,
    params={"kp_range": (0.85, 1.15), "kd_range": (0.6, 1.6), "operation": "scale",
            "asset_cfg": SceneEntityCfg("robot", actuator_names=(".*",))},
  )
  cfg.events["body_mass"] = EventTermCfg(
    mode="startup", func=dr.pseudo_inertia,  # mass and inertia together, about +-10%
    params={"alpha_range": (-0.05, 0.05), "asset_cfg": SceneEntityCfg("robot", body_names=MASS_BODIES)},
  )
  cfg.events["joint_friction"] = EventTermCfg(
    mode="startup", func=dr.joint_friction,
    params={"ranges": (0.0, 0.3), "operation": "abs",
            "asset_cfg": SceneEntityCfg("robot", joint_names=POLICY_JOINT_REGEX)},
  )
  cfg.events["joint_armature"] = EventTermCfg(
    mode="startup", func=dr.joint_armature,
    params={"ranges": (0.8, 1.2), "operation": "scale",
            "asset_cfg": SceneEntityCfg("robot", joint_names=POLICY_JOINT_REGEX)},
  )
  push = cfg.events["push_robot"].params["velocity_range"]
  push.update({"x": (-0.4, 0.4), "y": (-0.4, 0.4), "z": (-0.2, 0.2),
               "roll": (-0.3, 0.3), "pitch": (-0.3, 0.3), "yaw": (-0.5, 0.5)})

  # Rewards: mjlab's velocity rewards, pointed at the K1's bodies/sites/joints.
  cfg.rewards["upright"].params["asset_cfg"].body_names = ("trunk",)
  cfg.rewards["body_ang_vel"].params["asset_cfg"].body_names = ("trunk",)
  cfg.rewards["body_ang_vel"].weight = -0.05
  cfg.rewards["angular_momentum"].weight = -0.02
  for name in ("foot_clearance", "foot_slip"):
    cfg.rewards[name].params["asset_cfg"].site_names = FOOT_SITES
  for name in ("foot_clearance", "foot_swing_height"):
    cfg.rewards[name].params["target_height"] = 0.08
  # Smooth, calm motor commands (jittery commands are what make real robots shake).
  cfg.rewards["action_rate_l2"].weight = -0.2
  cfg.rewards["action_acc_l2"] = RewardTermCfg(func=envs_mdp.action_acc_l2, weight=-0.02)
  # Small bonus for every step it stays up, so the early (large) smoothness penalties never
  # make "fall quickly to end the episode" look attractive.
  cfg.rewards["alive"] = RewardTermCfg(func=envs_mdp.is_alive, weight=1.0)
  cfg.rewards["joint_acc_l2"] = RewardTermCfg(
    func=envs_mdp.joint_acc_l2, weight=-2.5e-7,
    params={"asset_cfg": SceneEntityCfg("robot", joint_names=POLICY_JOINT_REGEX)})
  cfg.rewards["joint_torques_l2"] = RewardTermCfg(
    func=envs_mdp.joint_torques_l2, weight=-1e-5,
    params={"asset_cfg": SceneEntityCfg("robot", actuator_names=POLICY_JOINT_REGEX)})
  cfg.rewards["pose"].params["asset_cfg"] = SceneEntityCfg("robot", joint_names=POLICY_JOINT_REGEX)
  cfg.rewards["pose"].params["std_standing"] = {".*": 0.05}
  cfg.rewards["pose"].params["std_walking"] = {
    r".*hip_pitch.*": 0.3, r".*hip_roll.*": 0.15, r".*hip_yaw.*": 0.15,
    r".*knee.*": 0.35, r".*ankle_pitch.*": 0.25, r".*ankle_roll.*": 0.1,
    r".*shoulder_pitch.*": 0.15, r".*shoulder_roll.*": 0.15, r".*elbow.*": 0.15,
  }
  cfg.rewards["pose"].params["std_running"] = {
    r".*hip_pitch.*": 0.5, r".*hip_roll.*": 0.2, r".*hip_yaw.*": 0.2,
    r".*knee.*": 0.6, r".*ankle_pitch.*": 0.35, r".*ankle_roll.*": 0.15,
    r".*shoulder_pitch.*": 0.5, r".*shoulder_roll.*": 0.2, r".*elbow.*": 0.35,
  }
  # Same safety limit as booster_deploy (it stops the policy beyond ~60 degrees tilt).
  cfg.terminations["fell_over"].params["limit_angle"] = math.radians(60.0)

  cfg.viewer.body_name = "trunk"
  cfg.viewer.distance = 2.0

  if play:
    cfg.episode_length_s = int(1e9)
    cfg.events.pop("push_robot", None)
    cfg.curriculum = {}
    twist.ranges.lin_vel_x = (-0.3, 1.0)
    twist.ranges.ang_vel_z = (-1.0, 1.0)
  return cfg


def k1_ppo_runner_cfg() -> RslRlOnPolicyRunnerCfg:
  return RslRlOnPolicyRunnerCfg(
    actor=RslRlModelCfg(
      hidden_dims=(512, 256, 128), activation="elu", obs_normalization=True,
      distribution_cfg={"class_name": "GaussianDistribution", "init_std": 1.0, "std_type": "scalar"},
    ),
    critic=RslRlModelCfg(hidden_dims=(512, 256, 128), activation="elu", obs_normalization=True),
    algorithm=RslRlPpoAlgorithmCfg(
      value_loss_coef=1.0, use_clipped_value_loss=True, clip_param=0.2, entropy_coef=0.01,
      num_learning_epochs=5, num_mini_batches=4, learning_rate=1.0e-3, schedule="adaptive",
      gamma=0.99, lam=0.95, desired_kl=0.01, max_grad_norm=1.0,
    ),
    experiment_name="k1_velocity_v2",
    logger="tensorboard",
    save_interval=100,
    num_steps_per_env=24,
    max_iterations=3_000,
  )


register_mjlab_task(
  task_id=TASK_ID,
  env_cfg=k1_flat_env_cfg(),
  play_env_cfg=k1_flat_env_cfg(play=True),
  rl_cfg=k1_ppo_runner_cfg(),
  runner_cls=VelocityOnPolicyRunner,
)


Writing /content/k1walk/k1_task.py


In [5]:
%%writefile /content/k1walk/train_k1.py
"""Train the Booster K1 walking policy with mjlab.

  python train_k1.py Mjlab-Velocity-Flat-Booster-K1 --env.scene.num-envs 4096
"""
import k1_task  # noqa: F401  (registers the K1 task)
from mjlab.scripts.train import main

if __name__ == "__main__":
    main()


Writing /content/k1walk/train_k1.py


In [6]:
%%writefile /content/k1walk/export_k1.py
"""Export a trained K1 walking policy for Booster's deploy tool, check it, and record a video.

  python export_k1.py --checkpoint logs/rsl_rl/k1_velocity/<run>/model_2999.pt
  python export_k1.py --checkpoint ... --video k1_walk.mp4 --seconds 12

Writes k1_my_walk.pt (TorchScript). booster_deploy feeds it 690 numbers (10 frames x 69, oldest
first, joints in booster_deploy's policy order) and expects 20 actions in that same joint order.
mjlab orders observations per term and joints in model order, so the exported module reorders
both ways. The script verifies this by computing the observation the way booster_deploy does,
from the simulator state, and comparing the actions with mjlab's own.
"""
from __future__ import annotations

import argparse
from collections import deque
from dataclasses import asdict

import numpy as np
import torch
import torch.nn as nn

import k1_task as K
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import MjlabOnPolicyRunner, RslRlVecEnvWrapper
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg, load_runner_cls

FRAME = 3 + 3 + 3 + 20 + 20 + 20  # 69 numbers per frame, booster_deploy layout


class DeployPolicy(nn.Module):
    """booster_deploy layout in -> mjlab policy -> booster_deploy joint order out."""

    def __init__(self, core: nn.Module, in_index: torch.Tensor, out_index: torch.Tensor):
        super().__init__()
        self.core = core
        self.register_buffer("in_index", in_index)
        self.register_buffer("out_index", out_index)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.reshape(-1, 690)
        actions = self.core(x[:, self.in_index])
        return actions[:, self.out_index].reshape(-1)


def build_index_maps(mjlab_joints: list[str]):
    """in_index[i] = which deploy input feeds mjlab input i; out_index[j] = mjlab action for deploy joint j."""
    deploy_pos = {name: i for i, name in enumerate(K.POLICY_JOINTS)}
    to_deploy = [deploy_pos[n] for n in mjlab_joints]          # mjlab joint k -> deploy slot
    # mjlab actor layout: per term, H frames (oldest first), each frame = term values.
    terms = [("ang", 3, 0), ("grav", 3, 3), ("cmd", 3, 6), ("pos", 20, 9), ("vel", 20, 29), ("act", 20, 49)]
    in_index = []
    for name, size, offset in terms:
        for frame in range(K.HISTORY):
            for k in range(size):
                slot = to_deploy[k] if size == 20 else k
                in_index.append(frame * FRAME + offset + slot)
    mjlab_pos = {n: k for k, n in enumerate(mjlab_joints)}
    out_index = [mjlab_pos[n] for n in K.POLICY_JOINTS]
    return torch.tensor(in_index, dtype=torch.long), torch.tensor(out_index, dtype=torch.long)


def make_env(play: bool, render: bool, device: str):
    cfg = load_env_cfg(K.TASK_ID, play=play) if play else load_env_cfg(K.TASK_ID)
    cfg.scene.num_envs = 1
    cfg.observations["actor"].enable_corruption = False
    return ManagerBasedRlEnv(cfg, device=device, render_mode="rgb_array" if render else None)


def deploy_frame(env, mjlab_joints, last_action_mjlab):
    """Build one 69-number frame exactly like booster_deploy's LocomotionPolicy.compute_observation."""
    robot = env.scene["robot"]
    names = list(robot.joint_names)
    ids = [names.index(n) for n in K.POLICY_JOINTS]
    q = robot.data.joint_pos_biased[0, ids]
    qd = robot.data.joint_vel[0, ids]
    q0 = robot.data.default_joint_pos[0, ids]
    ang = env.scene["robot/imu_ang_vel"].data[0]
    grav = robot.data.projected_gravity_b[0]
    cmd = env.command_manager.get_command("twist")[0]
    mj_pos = {n: k for k, n in enumerate(mjlab_joints)}
    act = last_action_mjlab[[mj_pos[n] for n in K.POLICY_JOINTS]]
    return torch.cat([ang, grav, cmd, q - q0, qd * K.JOINT_VEL_SCALE, act])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--checkpoint", required=True)
    ap.add_argument("--out", default="k1_my_walk.pt")
    ap.add_argument("--video", default=None, help="optional mp4 file to record the policy walking")
    ap.add_argument("--seconds", type=float, default=10.0)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    args = ap.parse_args()

    env = make_env(play=True, render=args.video is not None, device=args.device)
    agent_cfg = load_rl_cfg(K.TASK_ID)
    wrapped = RslRlVecEnvWrapper(env, clip_actions=agent_cfg.clip_actions)
    runner_cls = load_runner_cls(K.TASK_ID) or MjlabOnPolicyRunner
    runner = runner_cls(wrapped, asdict(agent_cfg), device=args.device)
    runner.load(args.checkpoint, load_cfg={"actor": True}, strict=True, map_location=args.device)

    core = runner.alg.get_policy().as_jit().to("cpu").eval()
    mjlab_joints = list(env.action_manager.get_term("joint_pos").target_names)
    in_index, out_index = build_index_maps(mjlab_joints)
    deploy_model = torch.jit.script(DeployPolicy(core, in_index, out_index))
    deploy_model.save(args.out)
    print(f"Saved {args.out}")

    # --- Run the policy, check the deploy-format model gives the same actions, record video ---
    policy = runner.get_inference_policy(device=args.device)
    obs = wrapped.get_observations()
    cmd_term = env.command_manager.get_term("twist")
    history: deque = deque(maxlen=K.HISTORY)
    last_action = torch.zeros(20, device=args.device)
    frames, max_diff, fell = [], 0.0, False
    steps = int(args.seconds / env.step_dt)
    def set_command(t):
        # Scripted commands for the test: stand, walk forward, curve, walk back, stand.
        phase = t * env.step_dt / args.seconds
        vx, wz = (0.0, 0.0) if phase < 0.1 else (0.5, 0.0) if phase < 0.5 else \
                 (0.3, 0.6) if phase < 0.7 else (-0.2, 0.0) if phase < 0.9 else (0.0, 0.0)
        cmd_term.vel_command_b[:] = torch.tensor([vx, 0.0, wz], device=cmd_term.vel_command_b.device)
        cmd_term.time_left[:] = 1e9
        for flag in ("is_heading_env", "is_standing_env", "is_world_env"):
            getattr(cmd_term, flag)[:] = False

    def push_frame():
        frame = deploy_frame(env, mjlab_joints, last_action).cpu()
        if not history:
            history.extend([frame] * K.HISTORY)  # booster_deploy's "repeat" history start
        else:
            history.append(frame)

    obs, _ = wrapped.reset()
    history.clear()
    push_frame()
    set_command(0)
    with torch.no_grad():
        for t in range(steps):
            action = policy(obs)
            deploy_action = deploy_model(torch.stack(list(history)).flatten())
            max_diff = max(max_diff, float((deploy_action - action[0].cpu()[out_index]).abs().max()))
            obs, _, dones, _ = wrapped.step(action)
            last_action = action[0].detach()
            if bool(dones[0]) and t < steps - 1:
                fell = True
                history.clear()
                last_action = torch.zeros(20, device=args.device)
            push_frame()           # same moment mjlab computes its observation
            set_command(t + 1)     # command for the next step
            if args.video:
                frames.append(env.render())

    print(f"Deploy-format check: max action difference over the run = {max_diff:.2e} "
          f"({'OK' if max_diff < 1e-4 else 'MISMATCH - do not deploy'})")
    print("Robot fell during the test run!" if fell else "Robot stayed up for the whole test run.")
    if args.video:
        import imageio
        imageio.mimsave(args.video, [np.asarray(f) for f in frames], fps=int(round(1 / env.step_dt)))
        print(f"Saved video {args.video}")


if __name__ == "__main__":
    main()


Writing /content/k1walk/export_k1.py


`k1_my_walk.py` is the small task file for **booster_deploy** (you'll copy it to your Mac at the end).

In [7]:
%%writefile /content/k1walk/k1_my_walk.py
"""booster_deploy task "k1_my_walk": runs YOUR walking policy trained with k1_task.py (mjlab).

Copy this file to:   booster_deploy/tasks/locomotion/robots/k1/k1_my_walk.py
Copy your policy to: booster_deploy/tasks/locomotion/robots/k1/models/k1_my_walk.pt

Simulation (Mac):  mjpython scripts/deploy.py --task k1_my_walk --mujoco
Real robot:        python3 scripts/deploy.py --task k1_my_walk

Same robot settings (default pose, PD gains, effort limits, joint order) as Booster's own
k1_walk task. Only the policy-specific options differ: the exported model already contains
the observation normalization, there is no action smoothing filter and no arm offset hack,
because the mjlab training used neither.
"""
from booster_deploy.utils.isaaclab.configclass import configclass
from booster_deploy.utils.registry import register_task

from ...locomotion import K1LocomotionPolicyCfg
from . import K1WalkControllerCfg


@configclass
class K1MyWalkTaskCfg(K1WalkControllerCfg):
    def __post_init__(self):
        super().__post_init__()
        self.policy = K1LocomotionPolicyCfg(
            policy_joint_names=self.policy.policy_joint_names,
            checkpoint_path="robots/k1/models/k1_my_walk.pt",
            actor_obs_history_length=10,
            history_init="repeat",
            obs_dof_vel_scale=0.1,
            action_scale=0.25,
            use_actor_module=False,
            action_filter=1.0,
            arm_action_fix_joint_name=None,
            arm_action_fix_offset=0.0,
        )


register_task("k1_my_walk", K1MyWalkTaskCfg())


Writing /content/k1walk/k1_my_walk.py


## 2. Smoke test (2 iterations, small)
Checks that everything installs and runs before you spend GPU hours.

In [8]:
%cd /content/k1walk
!PYTHONUNBUFFERED=1 MUJOCO_GL=egl python train_k1.py Mjlab-Velocity-Flat-Booster-K1 --env.scene.num-envs 256 --agent.max-iterations 2 --log-root /content/smoke 2>&1 | grep --line-buffered -E "Learning iteration|Mean reward|Mean episode|Error|Traceback" | tail -8

/content/k1walk
                             Learning iteration 0/2                              
                            Mean reward: -2.01
                    Mean episode length: 9.83
                             Learning iteration 1/2                              
                            Mean reward: -10.53
                    Mean episode length: 39.89


In [9]:
%cd /content/k1walk
!MUJOCO_GL=egl python train_k1.py Mjlab-Velocity-Flat-Booster-K1 --env.scene.num-envs 256 --agent.max-iterations 2 --log-root /content/smoke 2>&1 | tail -40

/content/k1walk
                            Mean reward: -10.41
                    Mean episode length: 39.59
                        Mean action std: 1.00
   Curriculum/command_vel/lin_vel_y_min: -0.3000
                 Episode_Reward/upright: 0.0259
           Episode_Termination/time_out: 0.4286
   Curriculum/command_vel/ang_vel_z_min: -0.8000
   Curriculum/command_vel/lin_vel_x_max: 0.6000
          Episode_Reward/dof_pos_limits: -0.0010
        Episode_Reward/joint_torques_l2: -0.0014
   Curriculum/command_vel/ang_vel_z_max: 0.8000
   Episode_Reward/track_linear_velocity: 0.0272
                   Episode_Reward/alive: 0.0391
          Metrics/angular_momentum_mean: 1.4682
                Episode_Reward/air_time: 0.0000
               Episode_Reward/foot_slip: -0.0008
            Episode_Reward/joint_acc_l2: -0.1951
            Metrics/twist/error_vel_yaw: 0.1437
          Episode_Reward/action_rate_l2: -0.3177
            Episode_Reward/soft_landing: -0.0000
             Metric

## 3. Train
Watch the curves in TensorBoard (refresh it now and then). Good signs: `Train/mean_reward` rises, `Train/mean_episode_length` approaches 1000 (= 20 s without falling), and `Metrics/twist/error_vel_xy` drops.

Every 100 iterations a checkpoint `model_XXXX.pt` is written to Drive.

In [10]:
%load_ext tensorboard
%tensorboard --logdir {LOG_ROOT}

<IPython.core.display.Javascript object>

In [11]:
%cd /content/k1walk
!PYTHONUNBUFFERED=1 MUJOCO_GL=egl python train_k1.py Mjlab-Velocity-Flat-Booster-K1 --env.scene.num-envs 4096 --agent.max-iterations 4000 --log-root "{LOG_ROOT}" 2>&1 | grep --line-buffered -E "Learning iteration|Mean reward|Mean episode length|error_vel|Total time|ETA|Error|Traceback"

Streaminguitvoer ingekort tot de laatste 5000 regels.
             Metrics/twist/error_vel_xy: 0.3756
                                    ETA: 0:18:09
                          Learning iteration 3167/4000                           
                            Mean reward: 109.90
                    Mean episode length: 1000.00
            Metrics/twist/error_vel_yaw: 0.5182
             Metrics/twist/error_vel_xy: 0.3694
                                    ETA: 0:18:08
                          Learning iteration 3168/4000                           
                            Mean reward: 108.16
                    Mean episode length: 983.41
            Metrics/twist/error_vel_yaw: 0.5112
             Metrics/twist/error_vel_xy: 0.3665
                                    ETA: 0:18:07
                          Learning iteration 3169/4000                           
                            Mean reward: 109.45
                    Mean episode length: 992.41
            Metrics/twis

### 3b. Resume after a disconnect (only if needed)
Continues from the newest checkpoint in your Drive.

In [12]:
%cd /content/k1walk
!PYTHONUNBUFFERED=1 MUJOCO_GL=egl python train_k1.py Mjlab-Velocity-Flat-Booster-K1 --env.scene.num-envs 4096 --agent.max-iterations 4000 --agent.resume True --log-root "{LOG_ROOT}" 2>&1 | grep --line-buffered -E "Loading model|Learning iteration|Mean reward|Mean episode length|error_vel|ETA|Error|Traceback"

/content/k1walk
[INFO]: Loading model checkpoint from: /content/drive/MyDrive/k1_walk/logs/k1_velocity_v2/2026-11-11_09-23-54/model_3999.pt
                          Learning iteration 3999/7999                           
                            Mean reward: 1.35
                    Mean episode length: 13.33
            Metrics/twist/error_vel_yaw: 0.0076
             Metrics/twist/error_vel_xy: 0.0071
                                    ETA: 2:20:54
                          Learning iteration 4000/7999                           
                            Mean reward: 3.91
                    Mean episode length: 35.79
            Metrics/twist/error_vel_yaw: 0.0194
             Metrics/twist/error_vel_xy: 0.0160
                                    ETA: 1:53:06
                          Learning iteration 4001/7999                           
                            Mean reward: 7.07
                    Mean episode length: 62.94
            Metrics/twist/error_vel_yaw: 0.03

## 4. Export for booster_deploy and watch it walk
This picks the newest checkpoint, writes `k1_my_walk.pt`, checks that the booster_deploy-format model gives the same actions as the trained policy (**must say OK**), and records a 12-second video: stand, walk forward, curve, walk backward, stand.

In [17]:
import glob, os
ckpts = sorted(glob.glob(f'{LOG_ROOT}/k1_velocity_v2/*/model_*.pt'), key=os.path.getmtime)
CKPT = ckpts[-1]
print('Using', CKPT)

Using /content/drive/MyDrive/k1_walk/logs/k1_velocity_v2/2026-11-11_10-53-05/model_4000.pt


In [14]:
%cd /content/k1walk
!PYTHONUNBUFFERED=1 MUJOCO_GL=egl python export_k1.py --checkpoint "{CKPT}" --video k1_walk.mp4 --seconds 12 2>&1 | grep --line-buffered -E "Saved|check|Robot|Error|Traceback"

/content/k1walk
Saved k1_my_walk.pt
Deploy-format check: max action difference over the run = 3.58e-07 (OK)
Robot stayed up for the whole test run.
Saved video k1_walk.mp4
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/OpenGL/error.py", line 230, in glCheckError
OpenGL.raw.EGL._errors.EGLError: EGLError(
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/OpenGL/error.py", line 230, in glCheckError
OpenGL.raw.EGL._errors.EGLError: EGLError(


In [15]:
from base64 import b64encode
from IPython.display import HTML
mp4 = b64encode(open('/content/k1walk/k1_walk.mp4', 'rb').read()).decode()
HTML(f'<video width=640 controls autoplay loop><source src="data:video/mp4;base64,{mp4}" type="video/mp4"></video>')

### Download the two files for your Mac

In [16]:
from google.colab import files
files.download('/content/k1walk/k1_my_walk.pt')
files.download('/content/k1walk/k1_my_walk.py')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5. On your Mac: test in Booster's simulator

```bash
conda activate booster
cd ~/booster/booster_deploy
cp ~/Downloads/k1_my_walk.py tasks/locomotion/robots/k1/
cp ~/Downloads/k1_my_walk.pt tasks/locomotion/robots/k1/models/
python scripts/deploy.py --list                       # should show k1_my_walk
mjpython scripts/deploy.py --task k1_my_walk --mujoco
```
Type speed commands like `0.5 0 0` in the terminal. booster_deploy uses a slightly different simulator setup than training, so this is a real robustness test.

## 6. On the robot
Only after it walks well in step 5, and **with firmware v1.7.2 or newer**:
copy `booster_deploy` (with the two new files) to the robot and run `python3 scripts/deploy.py --task k1_my_walk` (see the handover doc for the full steps). Use a harness or spotter for the first tries.